##This code look rule aggressive C = 1.5 and 3 strat 

support = 30, purity = 0.92 priority= best purity then freq 

RULE CONFIG → support=30 | purity=0.92 | priority=best_purity_then_freq
Macro F1: 0.7495120944712366
Rule coverage: 0.088 (921/10450)

support = 40  pruity 0.90 priority same at before 
RULE CONFIG → support=40 | purity=0.9 | priority=best_purity_then_freq

[Fold 1] mined rules: 82
Macro F1: 0.7486969315467812

Strategy agressive 
RULE CONFIG → support=30 | purity=0.95 | priority=best_purity_then_freq
[Fold 1] mined rules: 73
Macro F1: 0.7507806885172545

##by tuning c 
Samples after timestamp drop: 52247

===== FOLD 1 =====
Mined rules: 115

--- LogisticRegression C = 1.5 ---
Macro F1: 0.7503053925700244
Rule coverage: 0.106

--- LogisticRegression C = 2.0 ---
Macro F1: 0.747514418020134
Rule coverage: 0.106

--- LogisticRegression C = 3.0 ---
Macro F1: 0.7442488838301059
Rule coverage: 0.106

In [ ]:
# ============================================================
# TWO-STAGE RULE + LINEAR MODEL (RAW dev) — FULL PIPELINE
# - Drops rows with missing timestamp
# - Builds text = title + article (lowercased)
# - Numeric features: n_tokens, title_len, article_len, title_ratio, year, month, dow
# - Model: OHE(source) + word tfidf + char tfidf + scaled numeric -> LogisticRegression
# - Two-stage: mine pure token rules on TRAIN ONLY, override predictions on TEST when matched
# - Runs 1 fold (fast sanity) and evaluates 3 chosen strategies:
#   A) Baseline strong: support=30, purity=0.92, priority=best_purity_then_freq
#   B) Conservative:    support=40, purity=0.90, priority=best_purity_then_freq
#   C) Aggressive:      support=30, purity=0.95, priority=best_purity_then_freq
# - Also sweeps LogisticRegression C values for each strategy (default: [1.5])
# - Prints: Macro F1, rule coverage, rule-only F1, per-class recall, top matched tokens
#
# NOTE: This is "analysis-heavy" output but still runs in one file.
# ============================================================

import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Tuple, List

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
	f1_score,
	classification_report,
	confusion_matrix,
	recall_score
)

# ============================================================
# CONFIG
# ============================================================

DATA_PATH = "../data/raw/development.csv"

# CV config
N_SPLITS = 5
RANDOM_STATE = 42
USE_ONLY_FIRST_FOLD = True  # True = 1 fold only (fast). Set False for full CV.
APPLY_RULE_IF_FOUND = True  # two-stage override
PRINT_FULL_REPORT = False   # set True to print classification_report + confusion_matrix

# Rule tokenization config
TOKENIZATION = "split"      # keep it simple; you can change to regex-based later

# TF-IDF sizes (tune only if needed)
WORD_MAX_FEATURES = 250_000
CHAR_MAX_FEATURES = 300_000

# LogisticRegression sweep (you already saw best at ~1.5)
C_SWEEP = [1.5]  # optionally: [1.0, 1.5, 2.0, 3.0]

# The 3 strategies you asked for
STRATEGIES = [
	{
		"name": "A_baseline_strong",
		"min_support": 30,
		"purity": 0.92,
		"priority": "best_purity_then_freq",
	},
	{
		"name": "B_conservative",
		"min_support": 40,
		"purity": 0.90,
		"priority": "best_purity_then_freq",
	},
	{
		"name": "C_aggressive_ultrapure",
		"min_support": 30,
		"purity": 0.95,
		"priority": "best_purity_then_freq",
	},
]

# ============================================================
# DATA LOADING + CLEANING
# ============================================================

def load_and_prepare_raw(path: str) -> pd.DataFrame:
	df = pd.read_csv(path)

	# timestamp drop (your winning move)
	df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
	df = df[df["timestamp"].notna()].reset_index(drop=True)

	# basic fixes
	df["article"] = df["article"].fillna("").astype(str)
	df["title"] = df["title"].fillna("").astype(str)
	df["source"] = df["source"].fillna("").astype(str)

	# build text
	df["text"] = (df["title"] + " " + df["article"]).str.lower()

	# numeric features
	df["n_tokens"] = df["article"].str.split().str.len()
	df["title_len"] = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

	df["year"] = df["timestamp"].dt.year.astype(int)
	df["month"] = df["timestamp"].dt.month.astype(int)
	df["dow"] = df["timestamp"].dt.dayofweek.astype(int)

	return df


# ============================================================
# RULES: TOKENIZE, MINE, APPLY
# ============================================================

def tokenize_for_rules(text: str) -> List[str]:
	# super simple and FAST; keeps html-like tokens in place (good for your dataset)
	if TOKENIZATION == "split":
		return text.split()
	return text.split()


@dataclass
class RuleConfig:
	name: str
	min_support: int
	purity: float
	priority: str  # "best_purity_then_freq" or "freq_then_purity"


def mine_pure_rules(
	texts: pd.Series,
	labels: pd.Series,
	min_support: int,
	min_purity: float
) -> Tuple[Dict[str, int], Dict[str, Tuple[float, int]]]:
	"""
	Mine token -> class rules from TRAIN ONLY.

	Returns:
	- rule_token_to_class: token -> predicted class
	- rule_meta: token -> (purity, total_support)
	"""
	counts = defaultdict(lambda: Counter())

	# token presence, not counts (avoids long-article bias)
	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(y)] += 1

	rule_token_to_class = {}
	rule_meta = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < min_support:
			continue

		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= min_purity:
			rule_token_to_class[tok] = int(best_class)
			rule_meta[tok] = (float(purity), int(total))

	return rule_token_to_class, rule_meta


def apply_rules(
	texts: pd.Series,
	rule_token_to_class: Dict[str, int],
	rule_meta: Dict[str, Tuple[float, int]],
	priority: str
) -> Tuple[np.ndarray, List[str]]:
	"""
	For each text, if any rule token appears, assign class by best rule.
	Returns:
	- rule_pred: np.array with class labels or -1 if no rule matched
	- matched_token: list token matched (or None)
	"""
	rule_pred = np.full(len(texts), -1, dtype=int)
	matched_token = [None] * len(texts)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue

		if priority == "best_purity_then_freq":
			hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
		elif priority == "freq_then_purity":
			hits.sort(key=lambda t: (rule_meta[t][1], rule_meta[t][0]), reverse=True)
		else:
			# default fallback
			hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)

		best = hits[0]
		rule_pred[i] = int(rule_token_to_class[best])
		matched_token[i] = best

	return rule_pred, matched_token


# ============================================================
# MODEL
# ============================================================

def make_model(C: float) -> Pipeline:
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, 2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=WORD_MAX_FEATURES
			), "text"),

			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, 5),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=CHAR_MAX_FEATURES
			), "text"),

			("num", StandardScaler(), [
				"n_tokens", "title_len", "article_len",
				"title_ratio", "year", "month", "dow"
			]),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])


# ============================================================
# DIAGNOSTICS HELPERS
# ============================================================

def per_class_recall(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 7) -> Dict[int, float]:
	out = {}
	for c in range(n_classes):
		mask = (y_true == c)
		if mask.sum() == 0:
			out[c] = float("nan")
		else:
			out[c] = float((y_pred[mask] == c).mean())
	return out


def print_strategy_summary(
	strat_name: str,
	C: float,
	macro_f1: float,
	coverage: float,
	rule_only_f1: float,
	recalls: Dict[int, float],
	top_tokens: List[Tuple[str, int]]
):
	print("\n" + "=" * 80)
	print(f"STRATEGY: {strat_name} | C={C}")
	print("=" * 80)
	print(f"Macro F1: {macro_f1:.6f}")
	print(f"Rule coverage: {coverage:.3f}")
	if not np.isnan(rule_only_f1):
		print(f"Rule-only Macro F1 (matched subset): {rule_only_f1:.6f}")

	print("\nPer-class recall:")
	for c in sorted(recalls.keys()):
		print(f"  class {c}: recall={recalls[c]:.3f}")

	print("\nTop matched rule tokens:", top_tokens[:15])


# ============================================================
# RUN EXPERIMENTS (1 fold or full CV)
# ============================================================

def run_one_fold_experiments(df: pd.DataFrame):
	print("Samples after timestamp drop:", len(df))

	X = df[[
		"source", "text",
		"n_tokens", "title_len", "article_len",
		"title_ratio", "year", "month", "dow"
	]]
	y = df["label"].astype(int)

	skf = StratifiedKFold(
		n_splits=N_SPLITS,
		shuffle=True,
		random_state=RANDOM_STATE
	)

	for fold_id, (tr, te) in enumerate(skf.split(X, y), start=1):
		print(f"\n===== FOLD {fold_id} =====")

		X_tr, y_tr = X.iloc[tr].copy(), y.iloc[tr].copy()
		X_te, y_te = X.iloc[te].copy(), y.iloc[te].copy()

		# Evaluate each strategy
		for strat in STRATEGIES:
			rc = RuleConfig(
				name=strat["name"],
				min_support=strat["min_support"],
				purity=strat["purity"],
				priority=strat["priority"]
			)

			# ---- Mine rules on TRAIN ONLY
			rule_token_to_class, rule_meta = mine_pure_rules(
				X_tr["text"], y_tr,
				min_support=rc.min_support,
				min_purity=rc.purity
			)

			# ---- Apply for each C
			for C in C_SWEEP:
				model = make_model(C=C)
				model.fit(X_tr, y_tr)

				model_pred = model.predict(X_te)

				rule_pred, matched_token = apply_rules(
					X_te["text"],
					rule_token_to_class,
					rule_meta,
					priority=rc.priority
				)

				final_pred = model_pred.copy()
				mask = (rule_pred != -1)
				if APPLY_RULE_IF_FOUND:
					final_pred[mask] = rule_pred[mask]

				macro = f1_score(y_te, final_pred, average="macro")
				coverage = float(mask.mean())

				if mask.any():
					rule_only = f1_score(y_te[mask], final_pred[mask], average="macro")
				else:
					rule_only = float("nan")

				recalls = per_class_recall(y_te.to_numpy(), final_pred, n_classes=7)

				counter = Counter([t for t in matched_token if t is not None])
				top_tokens = counter.most_common(20)

				print_strategy_summary(
					strat_name=f"{rc.name} | support={rc.min_support} | purity={rc.purity} | priority={rc.priority}",
					C=C,
					macro_f1=macro,
					coverage=coverage,
					rule_only_f1=rule_only,
					recalls=recalls,
					top_tokens=top_tokens
				)

				if PRINT_FULL_REPORT:
					print("\nConfusion Matrix:\n", confusion_matrix(y_te, final_pred))
					print("\nReport:\n", classification_report(y_te, final_pred, digits=3))

		if USE_ONLY_FIRST_FOLD:
			break


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
	df = load_and_prepare_raw(DATA_PATH)
	run_one_fold_experiments(df)


Samples after timestamp drop: 52247

===== FOLD 1 =====

STRATEGY: A_baseline_strong | support=30 | purity=0.92 | priority=best_purity_then_freq | C=1.5
Macro F1: 0.752200
Rule coverage: 0.088
Rule-only Macro F1 (matched subset): 0.800031

Per-class recall:
  class 0: recall=0.708
  class 1: recall=0.817
  class 2: recall=0.849
  class 3: recall=0.669
  class 4: recall=0.922
  class 5: recall=0.640
  class 6: recall=0.795

Top matched rule tokens: [('/><img', 116), ('details.\\', 64), ('alt="democratic', 38), ('alt="republican', 33), ('afp', 31), ('(hollywood', 30), ('healthday', 29), ('inning', 29), ('sep.', 27), ('health)', 25), ('jun.', 21), ('night.</p><br', 20), ('vista', 18), ('rodham', 18), ('(pc', 17)]

STRATEGY: B_conservative | support=40 | purity=0.9 | priority=best_purity_then_freq | C=1.5
Macro F1: 0.751342
Rule coverage: 0.091
Rule-only Macro F1 (matched subset): 0.793506

Per-class recall:
  class 0: recall=0.707
  class 1: recall=0.815
  class 2: recall=0.851
  class 3: